<a href="https://colab.research.google.com/github/jou3487-bit/duplicados-articulos/blob/main/DEDUPLICACION_PH_NUMAR_BASE_ORACLE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# =====================================================================
# CELDA 1: CARGA, SINÓNIMOS Y NORMALIZACIÓN DE UNIDADES Y FRACCIONES
# =====================================================================
# @title Configuración Inicial y Normalización de Datos { display-mode: "form" }
!pip install thefuzz python-Levenshtein tqdm

import pandas as pd
import numpy as np
from thefuzz import fuzz
from tqdm import tqdm
import re
import time

# ── Diccionario de sinónimos técnicos ──────────────────────────
SINONIMOS = {
    r'\b(PULG|PULGADAS|PULGADA|IN|INCH|INCHES|\")\b': 'PULG',
    r'\b(MTS|METROS|METRO|MT|M)\b':                   'MT',
    r'\b(CMS|CENTIMETROS|CENTIMETRO|CM)\b':            'CM',
    r'\b(MMS|MILIMETROS|MILIMETRO|MM)\b':              'MM',
    r'\b(FTS|PIES|PIE|FT|FEET|FOOT|\')\b':            'FT',
    r'\b(YDS|YARDAS|YARDA|YD|YARDS)\b':               'YD',
    r'\b(KGS|KILOGRAMOS|KILOGRAMO|KG)\b':             'KG',
    r'\b(GRS|GRAMOS|GRAMO|GR|G)\b':                   'GR',
    r'\b(LBS|LIBRAS|LIBRA|LB)\b':                     'LB',
    r'\b(TON|TONELADAS|TONELADA|TN)\b':               'TON',
    r'\b(LTS|LITROS|LITRO|LT|L)\b':                   'LT',
    r'\b(GLS|GALONES|GALON|GL|GAL)\b':                'GL',
    r'\b(MLS|MILILITROS|MILILITRO|ML)\b':              'ML',
    r'\b(PSI|LB/IN2|LB/PULG2)\b':                    'PSI',
    r'\b(BAR|BARES)\b':                               'BAR',
    r'\b(AMP|AMPERES|AMPERE|AMPERIO)\b':              'AMP',
    r'\b(VOLT|VOLTIOS|VOLTIO|VT|V)\b':                'VOLT',
    r'\b(WATT|WATTS|VATIOS|W)\b':                     'WATT',
    r'\b(ACERO INOX|ACERO INOXIDABLE|INOX|SS|INOXIDABLE)\b': 'INOX',
    r'\b(GALVANIZADO|GALV|HDG)\b':                    'GALV',
    r'\b(PVC|POLICLORURO)\b':                         'PVC',
    r'\b(CPVC)\b':                                    'CPVC',
    r'\b(HDPE|POLIETILENO ALTA DENSIDAD)\b':          'HDPE',
    r'\b(NPT|NPTF|TAPERED)\b':                        'NPT',
    r'\b(BSP|BSPP|BSPT)\b':                           'BSP',
    r'\b(NUM|NUMERO|NO\.|NRO)\b':                     'NUM',
    r'\b(DIAM|DIAMETRO|DIA)\b':                       'DIAM',
    r'\b(LONG|LONGITUD|LARGO|LG)\b':                  'LONG',
    r'\b(PRES|PRESION)\b':                            'PRES',
    r'\b(MAX|MAXIMO|MAXIMA)\b':                       'MAX',
    r'\b(MIN|MINIMO|MINIMA)\b':                       'MIN',
    r'\b(TEMP|TEMPERATURA)\b':                        'TEMP',
}

FRACCIONES_COMUNES = {
    '1/8':  '0.1250', '2/8':  '0.2500', '3/8':  '0.3750', '4/8':  '0.5000',
    '5/8':  '0.6250', '6/8':  '0.7500', '7/8':  '0.8750', '1/4':  '0.2500',
    '3/4':  '0.7500', '1/2':  '0.5000', '1/16': '0.0625', '3/16': '0.1875',
    '5/16': '0.3125', '7/16': '0.4375', '9/16': '0.5625', '11/16': '0.6875',
    '13/16':'0.8125', '15/16': '0.9375', '1/32': '0.0313', '3/32': '0.0938',
    '1-1/8':'1.1250', '1-1/4':'1.2500', '1-3/8':'1.3750', '1-1/2':'1.5000',
    '1-3/4':'1.7500', '2-1/2':'2.5000', '3-1/2':'3.5000', '4-1/2':'4.5000',
}

def dividir_seguro(match):
    numerador = int(match.group(1))
    denominador = int(match.group(2))
    # Si el de abajo es cero, devolvemos el texto original intacto para evitar colapsos
    if denominador == 0:
        return match.group(0)
    return f"{numerador / denominador:.4f}"

def normalizar_fracciones_y_sinonimos(texto):
    # Paso A: Aplicar Sinónimos
    for patron, reemplazo in SINONIMOS.items():
        texto = re.sub(patron, reemplazo, texto)

    # Paso B: Normalizar decimales sin cero inicial (.5 → 0.5000)
    texto = re.sub(r'(?<!\d)\.(\d+)', lambda m: f"0.{m.group(1).ljust(4,'0')}", texto)

    # Paso C: Normalizar decimales con cero inicial (0.5 → 0.5000)
    texto = re.sub(r'\b(0\.\d+)\b', lambda m: f"{float(m.group(1)):.4f}", texto)

    # Paso D: Reemplazar fracciones conocidas
    for fraccion, decimal in sorted(FRACCIONES_COMUNES.items(), key=lambda x: len(x[0]), reverse=True):
        patron = re.escape(fraccion)
        texto = re.sub(rf'(?<!\d){patron}(?!\d)', decimal, texto)

    # Paso E: Cálculo dinámico PROTEGIDO contra división por cero
    texto = re.sub(r'\b(\d+)/(\d+)\b', dividir_seguro, texto)

    return texto

# --- CARGA Y PROCESAMIENTO ---
archivo_entrada = 'base_oracle.xlsx'
print("⏳ Cargando datos desde Excel...")
df = pd.read_excel(archivo_entrada)

df.columns = df.columns.astype(str).str.strip()
col_desc = [c for c in df.columns if 'DESC' in c.upper() or 'NAME' in c.upper()][0]
col_item = [c for c in df.columns if 'ITEM' in c.upper() or 'COD' in c.upper() or 'NUM' in c.upper()][0]
f_col = [c for c in df.columns if 'FAMILIA' in c.upper() and 'SUB' not in c.upper()][0]
sf_col = [c for c in df.columns if 'SUB' in c.upper() and 'FAM' in c.upper()][0]
sc_col = [c for c in df.columns if 'SUB' in c.upper() and 'CAT' in c.upper()][0]

print("🧹 Ejecutando estandarización dimensional y de sinónimos...")
df['Clean_Description'] = df[col_desc].astype(str).str.upper().str.strip()
df['Clean_Description'] = df['Clean_Description'].apply(lambda x: re.sub(r'\s+', ' ', x))

# Aplicamos la normalización protegida
df['Clean_Description'] = df['Clean_Description'].apply(normalizar_fracciones_y_sinonimos)

df['Bloque_Mantenimiento'] = df[f_col].astype(str).str.upper().str.strip() + " // " + \
                             df[sf_col].astype(str).str.upper().str.strip() + " // " + \
                             df[sc_col].astype(str).str.upper().str.strip()

print(f"📊 Base normalizada con {len(df):,} registros listos para comparación.")

⏳ Cargando datos desde Excel...
🧹 Ejecutando estandarización dimensional y de sinónimos...
📊 Base normalizada con 106,473 registros listos para comparación.


In [8]:
# =====================================================================
# CELDA 2: DEDUPLICACIÓN EN BASE A TEXTO NORMALIZADO Y EXCLUSIONES
# =====================================================================
UMBRAL_SIMILITUD = 90  # Un 90% es sumamente seguro ahora que las unidades son idénticas

df['GRUPO_ID'] = ""
df['DECISION'] = ""
df['REEMPLAZAR_POR'] = ""
df['SCORE_SIMILITUD'] = 100
grupo_contador = 1

LISTA_COLORES = {'NEGRO', 'BEIGE', 'BLANCO', 'ROJO', 'VERDE', 'AZUL', 'CAFE', 'NATURAL', 'AMARILLO', 'GRIS'}
LISTA_TALLAS = {'TALLA L', 'TALLA M', 'TALLA S', 'TALLA XL', 'TALLA XS'}
LISTA_POSICIONES = {'IZQ', 'DCH', 'DERECHO', 'IZQUIERDO'}

def extraer_numeros(texto):
    return set(re.findall(r'\d+\.?\d*', texto))

def encontrar_variante(texto, catalogo_variantes):
    return {variante for variante in catalogo_variantes if variante in texto}

def normalizar_plurales(texto):
    palabras = texto.split()
    palabras_limpias = [p[:-1] if p.endswith('S') and len(p) > 4 and p != 'GRATIS' else p for p in palabras]
    return " ".join(palabras_limpias)

print(f"🚀 Procesando bloques de mantenimiento...")
bloques_agrupados = df.groupby('Bloque_Mantenimiento')

for bloque, sub_df in tqdm(bloques_agrupados, desc="Analizando categorías"):
    indices = sub_df.index.tolist()

    while len(indices) > 0:
        idx_actual = indices.pop(0)
        if df.at[idx_actual, 'GRUPO_ID'] != "":
            continue

        desc_actual = df.at[idx_actual, 'Clean_Description']
        nums_actual = extraer_numeros(desc_actual)
        colores_actual = encontrar_variante(desc_actual, LISTA_COLORES)
        tallas_actual = encontrar_variante(desc_actual, LISTA_TALLAS)
        posicion_actual = encontrar_variante(desc_actual, LISTA_POSICIONES)

        desc_norm_actual = normalizar_plurales(desc_actual)
        posibles_duplicados = [idx_actual]

        for idx_comparar in indices:
            if df.at[idx_comparar, 'GRUPO_ID'] == "":
                desc_comparar = df.at[idx_comparar, 'Clean_Description']

                # REGLA 1: Validación estricta de números normalizados
                if nums_actual != extraer_numeros(desc_comparar):
                    continue

                # REGLA 2: Control de Colores
                if colores_actual != encontrar_variante(desc_comparar, LISTA_COLORES):
                    continue

                # REGLA 3: Control de Tallas
                if tallas_actual != encontrar_variante(desc_comparar, LISTA_TALLAS):
                    continue

                # REGLA 4: Control de Direcciones
                if posicion_actual != encontrar_variante(desc_comparar, LISTA_POSICIONES):
                    continue

                desc_norm_comparar = normalizar_plurales(desc_comparar)
                similitud = fuzz.ratio(desc_norm_actual, desc_norm_comparar)

                if similitud >= UMBRAL_SIMILITUD:
                    posibles_duplicados.append(idx_comparar)
                    df.at[idx_comparar, 'SCORE_SIMILITUD'] = similitud

        if len(posibles_duplicados) > 1:
            nombre_grupo = f"GRP_{grupo_contador:05d}"
            datos_grupo = df.loc[posibles_duplicados]

            idx_ganador = datos_grupo['Clean_Description'].str.len().idxmax()
            codigo_ganador = df.at[idx_ganador, col_item]

            for idx in posibles_duplicados:
                df.at[idx, 'GRUPO_ID'] = nombre_grupo
                if idx == idx_ganador:
                    df.at[idx, 'DECISION'] = "CONSERVAR (Maestro)"
                    df.at[idx, 'SCORE_SIMILITUD'] = 100
                else:
                    df.at[idx, 'DECISION'] = "REEMPLAZAR"
                    df.at[idx, 'REEMPLAZAR_POR'] = codigo_ganador
            grupo_contador += 1
        else:
            df.at[idx_actual, 'GRUPO_ID'] = "UNICO"
            df.at[idx_actual, 'DECISION'] = "CONSERVAR"

print("\n✨ Análisis finalizado con éxito.")

🚀 Procesando bloques de mantenimiento...


Analizando categorías: 100%|██████████| 1094/1094 [08:52<00:00,  2.05it/s]


✨ Análisis finalizado con éxito.


In [9]:
# =====================================================================
# CELDA 3: EXPORTACIÓN DEL EXCEL CORREGIDO
# =====================================================================
print("💾 Guardando el reporte estandarizado...")

df_final = df.drop(columns=['Clean_Description'])
df_final = df_final.sort_values(by=['GRUPO_ID', 'DECISION'], ascending=[True, False])

output_filename = 'resultado_deduplicacion_avanzado.xlsx'
df_final.to_excel(output_filename, index=False)

print(f"✅ ¡Todo listo! Descarga '{output_filename}' desde el panel lateral izquierdo de Colab.")

💾 Guardando el reporte estandarizado...
✅ ¡Todo listo! Descarga 'resultado_deduplicacion_avanzado.xlsx' desde el panel lateral izquierdo de Colab.
